In [ ]:
GENERATOR = 'Mistral-7B-Instruct-v0.3'
TAG = 'mistral7b'
LIMIT = None
REPO = 'nijatmammadov/R-CSA'
BRANCH = 'main'

In [ ]:
import os, base64, subprocess
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
ver = subprocess.run(['git', '--version'], capture_output=True, text=True).stdout.split()[2]
if tuple(int(x) for x in ver.split('.')[:2]) < (2, 31):
    raise SystemExit('git %s is too old for GIT_CONFIG_COUNT (needs 2.31)' % ver)
auth = base64.b64encode(('x-access-token:%s' % sec.get_secret('GITHUB_TOKEN')).encode()).decode()
os.environ.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                  GIT_CONFIG_VALUE_0='AUTHORIZATION: basic %s' % auth, GIT_TERMINAL_PROMPT='0')
try:
    os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
except Exception:
    pass
R = '/kaggle/working/repo'
url = 'https://github.com/%s' % REPO
if not os.path.exists(R):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, R], check=True)
else:
    subprocess.run(['git', '-C', R, 'remote', 'set-url', 'origin', url], check=True)
    subprocess.run(['git', '-C', R, 'pull', '-q', '--rebase', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', R, 'config', 'user.name', 'Nijat Mammadov'], check=True)
subprocess.run(['git', '-C', R, 'config', 'user.email', 'nijat.mmmadov@gmail.com'], check=True)
print(subprocess.run(['git', '-C', R, 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
os.chdir(R)

In [ ]:
!pip install -q vllm datasets openai python-dotenv
!pip uninstall -y -q torchaudio
!python -c "import torch; print('torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

In [ ]:
import sys, time, urllib.request
sys.path.insert(0, os.path.join(R, 'src'))
import pb_common as pc
MODEL = pc.model_for('vllm', GENERATOR)
os.environ['VLLM_BASE_URL'] = 'http://localhost:8000/v1'
log = open('/kaggle/working/vllm.log', 'w')
server = subprocess.Popen(['vllm', 'serve', MODEL, '--tensor-parallel-size', '2', '--dtype', 'half',
                           '--max-model-len', '16384', '--gpu-memory-utilization', '0.92'],
                          stdout=log, stderr=subprocess.STDOUT)
for i in range(240):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
        print('server up after %d s, serving %s' % (5 * i, MODEL))
        break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(5)
else:
    raise SystemExit('server did not come up in 20 minutes; see /kaggle/working/vllm.log')
if server.poll() is not None:
    print(open('/kaggle/working/vllm.log').read()[-3000:])
    raise SystemExit('vLLM exited')

In [ ]:
from openai import OpenAI
import math
cli = OpenAI(base_url=os.environ['VLLM_BASE_URL'], api_key='EMPTY')
r = cli.chat.completions.create(
    model=MODEL, max_tokens=1, temperature=0, logprobs=True, top_logprobs=10,
    messages=[{'role': 'user', 'content': 'Is 2 + 2 = 4 correct? Answer with one letter.\n(A) Correct\n(B) Incorrect'}])
top = {}
for t in r.choices[0].logprobs.content[0].top_logprobs:
    k = pc.clean_token(t.token).strip()
    top[k] = top.get(k, 0.0) + math.exp(t.logprob)
print({k: round(v, 4) for k, v in top.items()})
if 'A' not in top and 'B' not in top:
    raise SystemExit('neither A nor B among the top logprobs: check the judge prompt format before running')
print('OK: P(A) = %.3f' % top.get('A', 0.0))

In [ ]:
Q = 'data/dev/strategyqa_dev_questions.json'
SOLVED = 'data/dev/dev_%s_sqa2_solved.json' % TAG
B = 'results/dev/dev_%s_sqa' % TAG

def save(msg):
    subprocess.run(['git', 'add', '-A', 'data', 'results'], check=True)
    if subprocess.run(['git', 'diff', '--cached', '--quiet']).returncode == 0:
        print('nothing new to commit')
    else:
        subprocess.run(['git', 'commit', '-q', '-m', msg], check=True)
    subprocess.run(['git', 'pull', '-q', '--rebase', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'push', '-q', 'origin', BRANCH], check=True)
    print('pushed:', msg)

def run(cmd):
    print('$', cmd)
    rc = subprocess.run(cmd, shell=True).returncode
    if rc:
        raise SystemExit('failed (exit %d): %s' % (rc, cmd))

G = '--api vllm --generator %s' % GENERATOR
smoke = ' (smoke test)' if LIMIT else ''

In [ ]:
import json
print('LIMIT = %r -> solving %s questions' % (LIMIT, LIMIT or 200))
run('python src/pb_solve.py %s --questions %s --n %d --out %s' % (G, Q, LIMIT or 200, SOLVED))
d = json.load(open(SOLVED))
ok = [x for x in d if x['status'] == 'ok']
wrong = sum(not x['answer_correct'] for x in ok)
print('%d chains, %d ok, %d refusals/format errors, %d wrong answers (accuracy %.3f), %d without a Yes/No'
      % (len(d), len(ok), len(d) - len(ok), wrong, 1 - wrong / max(1, len(ok)), sum(x['answer'] is None for x in ok)))
save('%s StrategyQA dev: solved%s' % (GENERATOR, smoke))

In [ ]:
run('python src/css_fact.py %s --from-json %s --out %s_css.json --workers 6 --inner 8' % (G, SOLVED, B))
save('%s StrategyQA dev: css_fact.py (step probes, step sensitivity)%s' % (GENERATOR, smoke))
run('python src/pb_ptrue_full.py %s --from-json %s --out %s_ptrue_full.json --workers 8 --inner 8' % (G, SOLVED, B))
save('%s StrategyQA dev: P(TRUE) on full steps%s' % (GENERATOR, smoke))

In [ ]:
run('python src/pb_feng.py %s --from-json %s --out %s_feng.json --workers 6 --inner 8' % (G, SOLVED, B))
save('%s StrategyQA dev: Feng (2025) and Verbal Confidence%s' % (GENERATOR, smoke))
run('python src/pb_critic.py %s --from-json %s --out %s_critic.json --workers 16' % (G, SOLVED, B))
save('%s StrategyQA dev: ProcessBench critic%s' % (GENERATOR, smoke))
run('python src/pb_selfcheck.py %s --from-json %s --out %s_selfcheck.json --workers 4 --inner 8' % (G, SOLVED, B))
save('%s StrategyQA dev: SelfCheckGPT%s' % (GENERATOR, smoke))